# Switch to a local model (Ollama)

Ollama runs open-weight models in the codespace behind an OpenAI-compatible API: we change only `base_url` and `model`, then evaluate again.

In this notebook, we will:
1. Check that Ollama runs.
2. Point the OpenAI client to `localhost:11434/v1`.
3. Rerun the review classification on 20 reviews.
4. Compare quality, speed and cost with `gpt-6-luna`.

**Time:** about 30 minutes, self-study. Tip: a 4-core codespace is faster.

## Libraries and settings

`ask()` works exactly like `client.responses.create()` of the OpenAI library, with one difference: it saves every answer on disk, so running a cell again costs nothing. `ask_many()` sends many requests at the same time. You can read both functions in `src/llm_course/client.py`.

In [ ]:
# Libraries
import os
import re
import json
import time
import shutil
import platform
import subprocess
import urllib.request
import warnings
from datetime import datetime
from platform import python_version

import pandas as pd
from openai import OpenAI
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from llm_course import ask

# Ignore warnings
warnings.filterwarnings('ignore')

# Models to compare and list price of the API model (USD per 1M input tokens)
MODEL = 'gpt-6-luna'
PRICE_INPUT_PER_M = 0.10
LOCAL_MODEL = 'qwen3:1.7b'
OLLAMA_URL = 'http://localhost:11434/v1'

# Same labels and seed as in 03_Classify_Reviews
LABELS = ['positive', 'negative', 'mixed']
SEED = 42

# Check that the API key is available (never print the key itself)
print('API key found:', 'OPENAI_API_KEY' in os.environ)

# Show current working directory
print(os.getcwd())

## Step 1: Check that Ollama runs

Ollama is preinstalled in the codespace. If not, the next cell installs it (about 2 minutes) and starts the server.

In [ ]:
# Install Ollama if it is missing
if shutil.which('ollama') is None:
    print('Installing Ollama ...')
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

def ollama_running():
    """True if the Ollama server answers on port 11434."""
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2)
        return True
    except OSError:
        return False

# Start the server if it is not running
if not ollama_running():
    subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(3)

print('Ollama running:', ollama_running())

In [ ]:
# Download the model (about 1.4 GB, once)
subprocess.run(['ollama', 'pull', LOCAL_MODEL], check=True)

# List the local models (same as 'ollama list' in the terminal)
print(subprocess.run(['ollama', 'list'], capture_output=True, text=True).stdout)

## Step 2: Point the OpenAI client to localhost:11434/v1

Ollama needs no key, but the OpenAI client expects one: any text works.

In [ ]:
# Same library, different server
local = OpenAI(base_url=OLLAMA_URL, api_key='ollama')

def strip_thinking(text):
    """Remove <think>...</think> blocks that some local models write."""
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

r = local.chat.completions.create(
    model=LOCAL_MODEL,
    messages=[{'role': 'user', 'content': 'Name one typical dish in the IKEA restaurant. One sentence.'}],
)
print(strip_thinking(r.choices[0].message.content))

## Step 3: Rerun the review classification on 20 reviews

We draw the same sample as in `03_Classify_Reviews` and use its silver labels as the reference. If they are missing, we fall back to the star ratings (a noisy proxy).

In [ ]:
# Read all IKEA reviews (data from Moodle, see README: Data)
df = pd.read_csv(
    '../data/ikea_reviews.csv', sep=',', encoding='utf-8', parse_dates=['date']
)

# Same sample and split as in 03_Classify_Reviews; first 20 test reviews
sample = (
    df.groupby('stars')
    .sample(n=40, random_state=SEED)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)
_, test = train_test_split(sample, test_size=0.5, stratify=sample['stars'], random_state=SEED)
test = test.head(20).reset_index(drop=True)

# Reference labels
if os.path.exists('../data/processed/silver_labels.csv'):
    silver = pd.read_csv('../data/processed/silver_labels.csv')
    test = test.merge(silver, on='review_id', how='left')
    test['reference'] = test['silver']
    print('Reference: silver labels from 03_Classify_Reviews')
else:
    stars_to_label = {1: 'negative', 2: 'negative', 3: 'mixed', 4: 'positive', 5: 'positive'}
    test['reference'] = test['stars'].map(stars_to_label)
    print('Reference: star ratings (run 03_Classify_Reviews for silver labels)')

test[['stars', 'reference', 'text']].head()

In [ ]:
# A small schema: only the overall sentiment
schema = {
    'type': 'object',
    'properties': {'sentiment': {'type': 'string', 'enum': LABELS}},
    'required': ['sentiment'],
    'additionalProperties': False,
}

INSTRUCTIONS = (
    'You analyse customer reviews of IKEA. Label the overall sentiment of the '
    'review text as positive, negative or mixed. Watch out for irony. '
    'Answer in JSON.'
)

def parse(text):
    """Read the sentiment from a JSON answer; 'unparsed' if that fails."""
    try:
        return json.loads(strip_thinking(text))['sentiment']
    except (json.JSONDecodeError, KeyError, TypeError):
        return 'unparsed'

In [ ]:
# Local model: Chat Completions API with structured output
local_labels, local_seconds = [], []
for t in test['text']:
    start = time.time()
    r = local.chat.completions.create(
        model=LOCAL_MODEL,
        messages=[
            {'role': 'system', 'content': INSTRUCTIONS},
            {'role': 'user', 'content': f'<review>{t}</review>'},
        ],
        response_format={'type': 'json_schema',
                         'json_schema': {'name': 'review', 'schema': schema}},
    )
    local_seconds.append(time.time() - start)
    local_labels.append(parse(r.choices[0].message.content))

test['local'] = local_labels
print(test['local'].value_counts())

In [ ]:
# API model: Responses API with the same instructions and schema
# (use_cache=False: we want to measure the real speed)
api_labels, api_seconds, api_tokens = [], [], 0
for t in test['text']:
    start = time.time()
    r = ask(
        model=MODEL, instructions=INSTRUCTIONS, input=f'<review>{t}</review>',
        text={'format': {'type': 'json_schema', 'name': 'review',
                         'schema': schema, 'strict': True}},
        use_cache=False,
    )
    api_seconds.append(time.time() - start)
    api_labels.append(parse(r.output_text))
    api_tokens += r.usage.input_tokens

test['api'] = api_labels
print(test['api'].value_counts())

## Step 4: Compare quality, speed and cost

In [ ]:
comparison = pd.DataFrame({
    LOCAL_MODEL: {
        'accuracy': accuracy_score(test['reference'], test['local']),
        'seconds per review': sum(local_seconds) / len(test),
        'API cost (USD)': 0.0,
    },
    MODEL: {
        'accuracy': accuracy_score(test['reference'], test['api']),
        'seconds per review': sum(api_seconds) / len(test),
        'API cost (USD)': api_tokens / 1_000_000 * PRICE_INPUT_PER_M,
    },
}).T

comparison.round(4)

**Questions:**
- How much worse is the local model? Is it good enough for this task?
- What does "free" mean for the local model? (Hint: hardware, energy, setup, maintenance.)
- 1 million reviews per month: API or own server? Which data may not leave the company?

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')